# Sequential Sideband Reset Workflow


This notebook rewrites `examples/sequential_sideband_reset.py` as a guided workflow for a three-mode storage-transmon-readout model. The protocol uses repeated storage and readout sideband steps plus readout ringdown to remove storage excitations one level at a time.

The physical scenario is a reset recipe: map storage excitations through the transmon into the readout mode, let the readout dissipate, and repeat. This notebook is built on the reusable helper module in `examples/workflows/sequential_sideband_reset.py`, which is the canonical repo-side implementation of this workflow.


## Imports


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from tutorials.workflow_tutorial_support import configure_notebook_style

configure_notebook_style()

from cqed_sim import readout_photon_number, storage_photon_number, transmon_level_populations
from examples.workflows.sequential_sideband_reset import (
    SequentialSidebandResetCalibration,
    SequentialSidebandResetDevice,
    build_sideband_reset_frame,
    build_sideband_reset_model,
    build_sideband_reset_noise,
    run_sequential_sideband_reset,
)
from tutorials.tutorial_support import ns


## Physics / model definition


In [ ]:
device = SequentialSidebandResetDevice(
    readout_frequency_hz=8_596_222_556.078796,
    qubit_frequency_hz=6_150_358_764.4830475,
    storage_frequency_hz=5_240_932_800.0,
    readout_kappa_hz=4.156e6,
    qubit_anharmonicity_hz=-255_669_694.5244608,
    chi_storage_hz=-2_840_421.0,
    chi_readout_hz=-3.0e6,
    storage_gf_sideband_frequency_hz=6_803_533_628.0,
    storage_t1_s=250.0e-6,
    storage_t2_ramsey_s=150.0e-6,
)
calibration = SequentialSidebandResetCalibration(
    storage_sideband_rate_hz=8.0e6,
    readout_sideband_rate_hz=10.0e6,
    ef_rate_hz=12.0e6,
    ringdown_multiple=3.0,
)

model = build_sideband_reset_model(device, n_storage=5, n_readout=3)
frame = build_sideband_reset_frame(model)
noise = build_sideband_reset_noise(device)
initial_state = model.basis_state(0, 2, 0)


## Pulse / sequence construction


In [ ]:
pulse_dt_s = 0.25e-9
ringdown_dt_s = 4.0e-9
print("Reset protocol stages: storage sideband -> readout sideband -> ringdown, repeated for each storage excitation.")


## Simulation


In [ ]:
result = run_sequential_sideband_reset(
    model,
    initial_state,
    calibration=calibration,
    initial_storage_level=2,
    frame=frame,
    noise=noise,
    pulse_dt_s=pulse_dt_s,
    ringdown_dt_s=ringdown_dt_s,
)

stage_labels = [f"C{record.cycle_index}:{record.stage}" for record in result.stage_records]
stage_storage = np.asarray(
    [storage_photon_number(record.simulation.final_state) for record in result.stage_records],
    dtype=float,
)
stage_readout = np.asarray(
    [readout_photon_number(record.simulation.final_state) for record in result.stage_records],
    dtype=float,
)
stage_duration_ns = np.asarray([record.duration_s / ns for record in result.stage_records], dtype=float)
cycle_index = np.arange(1, result.cycle_final_storage_photon_number.size + 1, dtype=int)

print("Final transmon populations:", transmon_level_populations(result.final_state))
print("Final storage <n>:", storage_photon_number(result.final_state))
print("Final readout <n>:", readout_photon_number(result.final_state))


## Analysis / visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))

axes[0].plot(stage_storage, "o-", label="storage <n>", color="#4C78A8")
axes[0].plot(stage_readout, "o-", label="readout <n>", color="#E45756")
axes[0].set_xticks(np.arange(len(stage_labels)))
axes[0].set_xticklabels(stage_labels, rotation=45, ha="right")
axes[0].set_ylabel("Photon number")
axes[0].set_title("Stage-by-stage reset trajectory")
axes[0].legend(loc="best")

axes[1].plot(cycle_index, result.cycle_final_storage_photon_number, "o-", label="storage", color="#4C78A8")
axes[1].plot(cycle_index, result.cycle_final_readout_photon_number, "o-", label="readout", color="#E45756")
axes[1].set_xlabel("Reset cycle")
axes[1].set_ylabel("Photon number after cycle")
axes[1].set_title("Cycle summaries")
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()

print("Stage durations [ns]:", stage_duration_ns)


## Interpretation


The storage photon number drops after each cycle because the protocol repeatedly maps storage excitations into a dissipative readout channel. The readout population spikes during the dump step and then decays during ringdown.

This notebook is intentionally workflow-oriented rather than minimal. It shows how to use the repo-side helper module while keeping the physics assumptions explicit: matched rotating frame, effective sideband transitions, and Lindblad readout decay.


## Variations / exercises


- Change `initial_storage_level` to see how the number of cycles scales with the initial excitation.
- Increase `ringdown_multiple` to study the tradeoff between reset time and residual readout population.
- Run the same workflow with `noise=None` to separate coherent protocol errors from open-system degradation.
